# Análise de Rede, Colaboração, Geografia e Prestígio

Este notebook aborda as seguintes questões de pesquisa:

**RQ 2.1:** A proximidade no espaço semântico (distância de cosseno entre embeddings) é um preditor de coautoria? 

**RQ 2.2:** É possível identificar "escolas de pensamento" com base na instituição de doutorado? Pesquisadores formados na mesma instituição tendem a ser semanticamente mais próximos?

**RQ 2.3:** PPGs com notas mais altas (e.g., 6 e 7) possuem maior diversidade tópica (maior variância nos embeddings de seus membros)? Existe uma correlação entre a nota CAPES e a diversidade semântica?

## 1. Carregar Dados e Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import plotly.express as px
from sklearn.metrics.pairwise import cosine_similarity
import plotly.io as pio

pio.templates.default = "plotly_white"

# Carregar os dados com features
try:
    df = pd.read_parquet('../data/processed/featured_authors.parquet')
    # Converter a coluna de embedding de lista para array numpy
    embeddings = np.array(df['embedding'].tolist())
except FileNotFoundError:
    print("Arquivo 'featured_authors.parquet' não encontrado. Certifique-se de que a Fase 2 foi executada.")
    df = pd.DataFrame() # Cria um dataframe vazio para evitar erros posteriores
    embeddings = np.array([])

## 2. Análise de Coautoria vs. Proximidade Semântica (RQ 2.1)

**Nota:** Para esta análise, precisaríamos de uma etapa adicional na coleta de dados para extrair a rede de coautoria de cada pesquisador. Como não temos esses dados, vamos delinear a abordagem que seria usada.

In [ ]:
if not df.empty:
    # Exemplo hipotético de pares de autores
    # Em um cenário real, isso viria da rede de coautoria
    collaborators = [(0, 1), (2, 3)] # Pares de índices de autores que colaboraram
    non_collaborators = [(0, 3), (1, 2)] # Pares que não colaboraram
    
    def get_cosine_distances(author_pairs, embeddings):
        distances = []
        for i, j in author_pairs:
            sim = cosine_similarity(embeddings[i].reshape(1, -1), embeddings[j].reshape(1, -1))
            distances.append(1 - sim[0, 0]) # Distância = 1 - Similaridade
        return distances

    dist_collaborators = get_cosine_distances(collaborators, embeddings)
    dist_non_collaborators = get_cosine_distances(non_collaborators, embeddings)
    
    # Criar um DataFrame para o boxplot
    plot_df = pd.DataFrame({
        'Distância Cosseno': dist_collaborators + dist_non_collaborators,
        'Tipo': ['Colaboradores'] * len(dist_collaborators) + ['Não Colaboradores'] * len(dist_non_collaborators)
    })
    
    fig = px.box(plot_df, x='Tipo', y='Distância Cosseno', title='Distância Semântica entre Colaboradores e Não Colaboradores')
    fig.show()

## 3. Análise de "Escolas de Pensamento" (RQ 2.2)

In [ ]:
if not df.empty:
    # Visualizar pesquisadores por instituição de doutorado no espaço UMAP
    # Carregando as coordenadas UMAP geradas no notebook anterior
    # (Esta célula assume que o notebook 01 foi executado e os resultados salvos)
    # df_umap = pd.read_csv('../data/processed/umap_coords.csv') 
    # df = pd.merge(df, df_umap, on='id')
    
    fig = px.scatter(
        df,
        x='umap_x', # Assumindo que esta coluna existe
        y='umap_y', # Assumindo que esta coluna existe
        color='phd_institution_name',
        hover_data=['display_name'],
        title='Pesquisadores por Instituição de Doutorado'
    )
    fig.show()

## 4. Análise de Prestígio e Diversidade Tópica (RQ 2.3)

In [ ]:
if not df.empty:
    # Identificar PPGs com nota 6 e 7
    prestigious_ppgs = df[df['ppg_score'].isin([6, 7])]
    
    # Calcular a diversidade tópica (variância média dos embeddings) por PPG
    topical_diversity = []
    for ppg_code, group in df.groupby('gp_code'):
        if len(group) > 1:
            ppg_embeddings = np.array(group['embedding'].tolist())
            variance = np.mean(np.var(ppg_embeddings, axis=0))
            topical_diversity.append({
                'gp_code': ppg_code,
                'topical_diversity': variance,
                'ppg_score': group['ppg_score'].iloc[0]
            })
    
    diversity_df = pd.DataFrame(topical_diversity)
    
    # Correlacionar diversidade com a nota CAPES
    fig = px.scatter(
        diversity_df,
        x='topical_diversity',
        y='ppg_score',
        hover_data=['gp_code'],
        title='Diversidade Tópica vs. Nota CAPES',
        trendline='ols'
    )
    fig.show()